# RAG, Agentic RAG & GraphRAG — Complete Lab (Google Colab)

This notebook walks you through **three progressively advanced RAG architectures**:

| Part | Approach | What You'll Learn |
|------|----------|-------------------|
| **Part 1** | **Standard RAG** | Vector search + LLM generation using LCEL |
| **Part 2** | **Agentic RAG** | LLM agent that decides when to retrieve and can use multiple tools |
| **Part 3** | **GraphRAG** | Knowledge graph construction, community detection, local & global search |

### Tech Stack
- **LLM**: Google Gemini (`gemini-2.0-flash`) via `langchain-google-genai`
- **Embeddings**: HuggingFace `all-MiniLM-L6-v2`
- **Vector Store**: ChromaDB
- **Agent Framework**: LangGraph
- **Graph**: NetworkX + Graspologic (Leiden community detection)
- **Chain Composition**: LangChain Expression Language (LCEL)

In [ ]:
# Install required packages
!pip install -qU langchain langchain-google-genai langchain-chroma langchain-huggingface sentence-transformers

In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')

## Phase 1: Ingestion
Load and chunk the source document.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a dummy syllabus file for the lab
with open("course_data.txt", "w") as f:
    f.write("The course AI-2025 grading policy is: 50% Final Project, 30% Labs, 20% Midterm. The instructor is Dr. Smith.")

loader = TextLoader("course_data.txt")
docs = loader.load()

# Splitter - crucial for RAG
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
all_splits = splitter.split_documents(docs)

print(f"Split into {len(all_splits)} chunks")
for i, chunk in enumerate(all_splits):
    print(f"  Chunk {i}: {chunk.page_content[:80]}...")

## Phase 2: Indexing
Embed the chunks and store them in ChromaDB.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Using a free, high-quality open source embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create vector database in memory
vector_db = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings
)

print(f"Stored {vector_db._collection.count()} vectors in ChromaDB")

## Phase 3: Retrieval & Generation
Build the RAG chain using LCEL with Google Gemini.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Define the retriever (Fetch top 2 chunks)
retriever = vector_db.as_retriever(search_type="similarity", search_kwargs={"k": 2})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the provided context.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}"""
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")

In [ ]:
# --- TEST ---
query = "How is the course graded?"
print(f"Question: {query}")
answer = qa_chain.invoke(query)
print(f"Answer: {answer}")

In [ ]:
# Try another question
query = "Who is the instructor?"
print(f"Question: {query}")
answer = qa_chain.invoke(query)
print(f"Answer: {answer}")

---

# Part 2: Agentic RAG

## What is Agentic RAG?

**Standard RAG** follows a fixed pipeline: retrieve → stuff into prompt → generate. It has no ability to *reason* about whether the retrieved context is sufficient, ask follow-up questions, or use external tools.

**Agentic RAG** wraps the RAG pipeline inside an **AI agent** that can:
1. **Decide** whether it needs to retrieve documents at all (maybe it already knows the answer)
2. **Evaluate** retrieved chunks — if they're not relevant, it can retry with a different query
3. **Use tools** — e.g., a calculator, web search, or database lookup alongside the vector store
4. **Route** questions to different retrieval sources depending on the topic

### Key Concepts
| Concept | Description |
|---------|-------------|
| **Agent** | An LLM that can choose which *tools* to call and in what order |
| **Tool** | A function the agent can invoke (e.g., vector search, web search, calculator) |
| **Tool Calling** | The LLM outputs a structured request to call a tool, rather than a plain text answer |
| **ReAct Loop** | **Re**ason + **Act** — the agent thinks step-by-step, calling tools as needed, then synthesizes a final answer |

### Why Agentic RAG?
- **Adaptive retrieval**: The agent decides *when* and *what* to retrieve
- **Self-correction**: If the first retrieval doesn't answer the question, the agent can reformulate and retry
- **Multi-source**: The agent can query multiple knowledge bases or APIs in a single turn
- **Tool use**: Combine retrieval with computation (e.g., "What's 20% of the midterm weight?" → retrieves grading policy, then calculates)

### Architecture Diagram
```
User Question
      │
      ▼
┌──────────┐
│   Agent   │ ◄── LLM with tool-calling capability
│  (ReAct)  │
└────┬─────┘
     │  Decides which tool(s) to use
     ▼
┌─────────────────────────────────┐
│         Available Tools          │
│  ┌───────────┐  ┌────────────┐  │
│  │  Vector    │  │ Calculator │  │
│  │  Search    │  │            │  │
│  └───────────┘  └────────────┘  │
└─────────────────────────────────┘
     │
     ▼
  Final Answer (synthesized from tool outputs)
```

In [ ]:
# Install langgraph for agent orchestration
!pip install -qU langgraph

### Step 1: Define Tools for the Agent

In Agentic RAG, we convert our vector store retriever into a **tool** — a callable function that the agent can choose to invoke. We also add a simple **calculator tool** to show multi-tool capability.

Key point: The agent **doesn't always call every tool**. It reads the question, reasons about what's needed, and only calls the tools it thinks will help.

In [ ]:
from langchain_core.tools import tool

# --- Tool 1: Vector Store Search ---
# We wrap our existing retriever as a tool the agent can call.
# The docstring is critical — the agent reads it to decide WHEN to use this tool.

@tool
def search_course_info(query: str) -> str:
    """Search the course syllabus and policies. Use this tool for questions about
    grading, assignments, the instructor, course schedule, or any course-related info."""
    docs = retriever.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)


# --- Tool 2: Calculator ---
# A simple tool to demonstrate multi-tool agents.
# The agent can retrieve grading weights AND compute percentages in one turn.

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Use this for any arithmetic calculations.
    Input should be a valid Python math expression like '50 + 30 + 20' or '0.5 * 100'."""
    try:
        result = eval(expression, {"__builtins__": {}})  # restricted eval for safety
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# Collect all tools into a list
tools = [search_course_info, calculator]

print("Tools defined:")
for t in tools:
    print(f"  - {t.name}: {t.description[:80]}...")

### Step 2: Create the Agent with LangGraph

We use **LangGraph** to build a **ReAct agent** — an agent that follows the Reason-Act cycle:

1. **Reason**: The LLM looks at the question and decides which tool to call (or whether to answer directly)
2. **Act**: The chosen tool is executed and its output is fed back to the LLM
3. **Repeat**: The LLM decides if it has enough information or if it needs to call another tool
4. **Answer**: Once satisfied, the LLM generates the final answer

The `create_react_agent` function from LangGraph handles this loop automatically.

In [ ]:
from langgraph.prebuilt import create_react_agent

# Create a ReAct agent that can use our tools
# The LLM must support tool calling (Gemini does)
agent = create_react_agent(
    model=llm,       # Same Gemini model from Part 1
    tools=tools,     # The tools we defined above
)

print("Agentic RAG agent created!")

### Step 3: Test the Agentic RAG

Let's test with different types of questions to see the agent's behavior:

1. **Retrieval question** — The agent should call `search_course_info`
2. **Retrieval + calculation** — The agent should call both tools
3. **General knowledge** — The agent may answer directly without any tool

Watch the agent's reasoning in the output — you'll see it decide which tools to use!

In [ ]:
# Helper to run the agent and display results clearly
def ask_agent(question: str):
    print(f"Question: {question}")
    print("-" * 60)
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})

    # Show each step the agent took
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  🔧 Agent calls tool: {tc['name']}(\"{tc['args']}\")")
        elif role == "ToolMessage":
            print(f"  📄 Tool returned: {msg.content[:150]}...")
        elif role == "AIMessage" and msg.content:
            print(f"\n✅ Final Answer: {msg.content}")
    print()

In [ ]:
# Test 1: Pure retrieval — agent should use search_course_info
ask_agent("What is the grading policy for the course?")

In [ ]:
# Test 2: Retrieval + calculation — agent should use BOTH tools
ask_agent("If the final project is worth 50% and I scored 85 out of 100, how many points does that contribute to my final grade?")

In [ ]:
# Test 3: General knowledge — agent may answer directly without tools
ask_agent("What is the capital of France?")

### Agentic RAG — Key Takeaways

| What we saw | Why it matters |
|---|---|
| The agent chose which tools to call | Unlike basic RAG, the agent doesn't blindly retrieve — it **reasons** first |
| It combined retrieval + calculation | Multi-tool agents can **compose capabilities** in a single turn |
| It answered general questions directly | The agent knows when retrieval is unnecessary, saving time and tokens |

> **Think of it this way**: Basic RAG is like a librarian who always fetches books before answering. Agentic RAG is like a smart assistant who first thinks "Do I already know this?" and only goes to the library when needed.

---

# Part 3: GraphRAG

## What is GraphRAG?

**Standard RAG** retrieves isolated text chunks based on vector similarity. This works well for factual, localized questions ("What is the grading policy?") but struggles with questions that require **connecting information across multiple chunks** or understanding **relationships** between entities.

**GraphRAG** (Graph-based Retrieval-Augmented Generation) solves this by:
1. **Extracting entities and relationships** from the text to build a **knowledge graph**
2. **Detecting communities** of related entities (clusters of connected information)
3. **Generating summaries** at different levels of the graph hierarchy
4. Using these **structured summaries** for retrieval instead of (or alongside) raw text chunks

### The Problem GraphRAG Solves

Consider a large document about a university. Standard RAG might retrieve:
- Chunk A: "Dr. Smith teaches AI-2025"
- Chunk B: "The AI department is in the Engineering building"

But if you ask **"Where does Dr. Smith's department meet?"**, standard RAG might not connect these two pieces. GraphRAG builds a graph: `Dr. Smith → teaches → AI-2025 → part_of → AI Department → located_in → Engineering Building`, making this connection explicit.

### How GraphRAG Works (Step by Step)

```
         Source Documents
               │
               ▼
    ┌─────────────────────┐
    │  1. Entity & Relation │  ← LLM extracts (Subject, Predicate, Object) triples
    │     Extraction        │     e.g., ("Dr. Smith", "teaches", "AI-2025")
    └──────────┬────────────┘
               │
               ▼
    ┌─────────────────────┐
    │  2. Knowledge Graph   │  ← Entities become nodes, relations become edges
    │     Construction      │
    └──────────┬────────────┘
               │
               ▼
    ┌─────────────────────┐
    │  3. Community         │  ← Leiden algorithm groups related entities into
    │     Detection         │     clusters (communities)
    └──────────┬────────────┘
               │
               ▼
    ┌─────────────────────┐
    │  4. Community         │  ← LLM summarizes each community
    │     Summarization     │     "This community covers the AI department..."
    └──────────┬────────────┘
               │
               ▼
    ┌─────────────────────┐
    │  5. Query Answering   │  ← Two modes:
    │     (Local or Global) │     Local: search specific entities
    └───────────────────────┘     Global: search community summaries
```

### Local Search vs. Global Search

| Mode | Best For | How It Works |
|------|----------|-------------|
| **Local Search** | Specific questions about particular entities | Finds the entity in the graph, traverses its neighborhood, retrieves connected information |
| **Global Search** | Broad, thematic questions ("What are the main themes?") | Searches across community summaries at different hierarchy levels |

### Reference
- **Paper**: [From Local to Global: A Graph RAG Approach to Query-Focused Summarization](https://arxiv.org/pdf/2404.16130)
- **GitHub**: [microsoft/graphrag](https://github.com/microsoft/graphrag)

In [ ]:
# Install GraphRAG dependencies
# networkx: graph data structure
# graspologic: community detection (Leiden algorithm)
!pip install -qU networkx graspologic

### Step 1: Create a Richer Dataset

GraphRAG shines when there are **entities and relationships** to extract. Let's create a richer document with multiple people, courses, departments, and locations that are interconnected.

In [ ]:
# A richer document with multiple entities and relationships
university_docs = [
    "Dr. Smith teaches the AI-2025 course and leads the Machine Learning Research Lab. "
    "The ML Lab is located in the Engineering Building, Room 302.",

    "Dr. Chen is the chair of the Computer Science Department. She oversees both the "
    "AI-2025 and Databases-3010 courses. The CS Department is housed in the Engineering Building.",

    "The Engineering Building was renovated in 2023 and contains the ML Lab, the Robotics Lab, "
    "and the CS Department offices. It is located on the North Campus.",

    "Professor Garcia teaches Databases-3010 and collaborates with Dr. Smith on the "
    "Knowledge Graphs research project. They published a paper together in 2024.",

    "The AI-2025 course covers machine learning, deep learning, and natural language processing. "
    "It requires students to complete a final project worth 50% of their grade.",

    "Dr. Kim runs the Robotics Lab and teaches Robotics-4020. The Robotics Lab "
    "has 15 robotic arms and 3 autonomous vehicles for student experiments.",

    "The North Campus also houses the Physics Building and the Student Union. "
    "The Engineering Building is connected to the Physics Building by a skybridge.",

    "Dr. Smith and Dr. Kim are co-PIs on a National Science Foundation grant "
    "studying autonomous navigation using reinforcement learning.",
]

print(f"Created {len(university_docs)} document chunks about a university")
for i, doc in enumerate(university_docs):
    print(f"  Doc {i}: {doc[:70]}...")

### Step 2: Entity & Relationship Extraction using an LLM

This is the core of GraphRAG — we ask the LLM to read each chunk and extract **entities** (people, places, courses, labs) and **relationships** between them as `(subject, predicate, object)` triples.

This step replaces the simple embedding-based indexing from standard RAG.

In [ ]:
import json

# Prompt the LLM to extract entities and relationships from each document
extraction_prompt = ChatPromptTemplate.from_template(
    """Extract all entities and relationships from the following text.

Return a JSON object with two keys:
- "entities": a list of strings (names of people, places, departments, courses, labs, buildings, etc.)
- "relationships": a list of [subject, predicate, object] triples

Example output:
{{"entities": ["Dr. Smith", "AI-2025", "ML Lab"], "relationships": [["Dr. Smith", "teaches", "AI-2025"], ["Dr. Smith", "leads", "ML Lab"]]}}

Text: {text}

Return ONLY valid JSON, no other text."""
)

extraction_chain = extraction_prompt | llm | StrOutputParser()

# Extract from all documents
all_entities = set()
all_relationships = []

print("Extracting entities & relationships from each document...\n")
for i, doc_text in enumerate(university_docs):
    raw = extraction_chain.invoke({"text": doc_text})
    try:
        # Strip markdown code fences if present
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0]
        data = json.loads(cleaned)
        entities = data.get("entities", [])
        relationships = data.get("relationships", [])

        all_entities.update(entities)
        all_relationships.extend(relationships)

        print(f"  Doc {i}: {len(entities)} entities, {len(relationships)} relationships")
    except json.JSONDecodeError:
        print(f"  Doc {i}: ⚠️ Failed to parse LLM output, skipping")

print(f"\nTotal unique entities: {len(all_entities)}")
print(f"Total relationships: {len(all_relationships)}")

### Step 3: Build the Knowledge Graph

Now we construct an actual graph data structure using **NetworkX**. Each entity becomes a **node**, and each relationship becomes a **directed edge** with a label.

This is the key differentiator from standard RAG — instead of a flat collection of text chunks, we have a **structured representation** of the knowledge.

In [ ]:
import networkx as nx

# Build a directed graph from extracted entities and relationships
G = nx.Graph()  # Undirected for community detection

# Add all entities as nodes
for entity in all_entities:
    G.add_node(entity)

# Add relationships as edges
for subj, pred, obj in all_relationships:
    G.add_edge(subj, obj, label=pred)

print(f"Knowledge Graph built!")
print(f"  Nodes (entities): {G.number_of_nodes()}")
print(f"  Edges (relationships): {G.number_of_edges()}")
print(f"\nSample edges:")
for u, v, data in list(G.edges(data=True))[:8]:
    print(f"  {u} --[{data.get('label', '?')}]--> {v}")

### Step 4: Community Detection

The **Leiden algorithm** groups tightly connected nodes into **communities**. In a university knowledge graph, you might get communities like:
- Community 1: AI/ML cluster (Dr. Smith, AI-2025, ML Lab)
- Community 2: Database/CS cluster (Dr. Chen, Prof. Garcia, Databases-3010)
- Community 3: Location cluster (Engineering Building, North Campus)

Each community is later summarized, enabling **global search** over themes rather than individual facts.

In [ ]:
from graspologic.partition import hierarchical_leiden

# Run hierarchical Leiden community detection
# This returns communities at multiple levels of granularity
community_results = hierarchical_leiden(G, max_cluster_size=10)

# Organize: map each node to its community
node_to_community = {}
for result in community_results:
    node_to_community[result.node] = result.cluster

# Group nodes by community
communities = {}
for node, comm_id in node_to_community.items():
    communities.setdefault(comm_id, []).append(node)

print(f"Found {len(communities)} communities:\n")
for comm_id, members in sorted(communities.items()):
    print(f"  Community {comm_id} ({len(members)} members):")
    for m in members:
        print(f"    - {m}")

### Step 5: Community Summarization

For each community of entities, we ask the LLM to generate a **summary** that describes the theme and key relationships within that community. These summaries are what **Global Search** queries against.

In [ ]:
summary_prompt = ChatPromptTemplate.from_template(
    """You are analyzing a knowledge graph community. Below are the entities in this community
and their relationships.

Entities: {entities}

Relationships:
{relationships}

Write a concise 2-3 sentence summary that describes what this community is about,
what the key entities are, and how they relate to each other."""
)

summary_chain = summary_prompt | llm | StrOutputParser()

# Generate summaries for each community
community_summaries = {}

print("Generating community summaries...\n")
for comm_id, members in sorted(communities.items()):
    # Get all edges within this community
    comm_edges = []
    for u, v, data in G.edges(data=True):
        if u in members and v in members:
            comm_edges.append(f"  {u} --[{data.get('label', '?')}]--> {v}")

    if not comm_edges:
        comm_edges = ["  (No internal relationships found)"]

    summary = summary_chain.invoke({
        "entities": ", ".join(members),
        "relationships": "\n".join(comm_edges)
    })

    community_summaries[comm_id] = {
        "members": members,
        "summary": summary
    }

    print(f"Community {comm_id}:")
    print(f"  Members: {', '.join(members)}")
    print(f"  Summary: {summary}\n")

### Step 6: GraphRAG Querying — Local Search

**Local Search** starts from a specific entity in the graph and traverses its neighborhood to gather context. It's ideal for questions about specific people, places, or things.

This is like asking "Tell me everything connected to Dr. Smith" — the graph makes it easy to gather all related information structurally, even if that information was spread across many different text chunks.

In [ ]:
def local_search(graph, query_entity: str, depth: int = 2) -> str:
    """
    Local Search: Find an entity in the graph, traverse its neighborhood,
    and return all connected entities and relationships as context.

    Args:
        graph: NetworkX graph
        query_entity: The entity to search for (fuzzy match)
        depth: How many hops from the entity to traverse
    """
    # Fuzzy match: find the closest node name
    matched_node = None
    query_lower = query_entity.lower()
    for node in graph.nodes():
        if query_lower in node.lower():
            matched_node = node
            break

    if not matched_node:
        return f"Entity '{query_entity}' not found in the knowledge graph."

    # Get the subgraph within 'depth' hops
    neighbors = set()
    current_level = {matched_node}
    for _ in range(depth):
        next_level = set()
        for node in current_level:
            next_level.update(graph.neighbors(node))
        neighbors.update(next_level)
        current_level = next_level

    neighbors.add(matched_node)

    # Gather all edges in this neighborhood
    context_parts = [f"Entity: {matched_node}", f"Connected entities within {depth} hops:", ""]
    for u, v, data in graph.edges(data=True):
        if u in neighbors and v in neighbors:
            context_parts.append(f"  {u} --[{data.get('label', 'related_to')}]--> {v}")

    return "\n".join(context_parts)


# Test local search
print("=" * 60)
print("LOCAL SEARCH: Finding context for 'Dr. Smith'")
print("=" * 60)
local_context = local_search(G, "Smith", depth=2)
print(local_context)

In [ ]:
# Now use the local search context to answer a question
local_answer_prompt = ChatPromptTemplate.from_template(
    """Use the following knowledge graph context to answer the question.
Only use information from the provided context.

Knowledge Graph Context:
{context}

Question: {question}"""
)

local_qa_chain = local_answer_prompt | llm | StrOutputParser()

question = "What does Dr. Smith work on and where is the lab located?"
print(f"\nQuestion: {question}")
answer = local_qa_chain.invoke({"context": local_context, "question": question})
print(f"Answer: {answer}")

### Step 7: GraphRAG Querying — Global Search

**Global Search** queries the **community summaries** rather than individual entities. It's ideal for broad, thematic questions like:
- "What are the main research areas at this university?"
- "How are the departments connected?"

This is what makes GraphRAG unique — standard RAG cannot answer these "big picture" questions well because no single chunk contains the full answer.

In [ ]:
def global_search(community_summaries: dict) -> str:
    """
    Global Search: Concatenate all community summaries to provide
    a high-level overview of the entire knowledge base.
    """
    context_parts = []
    for comm_id, data in sorted(community_summaries.items()):
        context_parts.append(
            f"Community {comm_id} (Members: {', '.join(data['members'])}):\n"
            f"  {data['summary']}"
        )
    return "\n\n".join(context_parts)


# Build global context from all community summaries
global_context = global_search(community_summaries)
print("Global context (all community summaries):\n")
print(global_context)

In [ ]:
# Answer a broad thematic question using community summaries
global_answer_prompt = ChatPromptTemplate.from_template(
    """You are answering a broad question using high-level summaries of knowledge communities.
Each community represents a cluster of related entities in the knowledge graph.

Community Summaries:
{context}

Question: {question}

Provide a comprehensive answer that synthesizes information across all relevant communities."""
)

global_qa_chain = global_answer_prompt | llm | StrOutputParser()

# Test: A broad question that requires connecting information across the whole graph
question = "What are the main research areas and how are the faculty connected to each other?"
print(f"Question: {question}\n")
answer = global_qa_chain.invoke({"context": global_context, "question": question})
print(f"Answer: {answer}")

In [ ]:
# Another global search test - this question requires understanding the overall structure
question = "Give me an overview of the university's Engineering Building and everything that happens there."
print(f"Question: {question}\n")
answer = global_qa_chain.invoke({"context": global_context, "question": question})
print(f"Answer: {answer}")

### Step 8: Visualize the Knowledge Graph

Let's visualize the graph to see the entities and relationships we extracted. This helps build intuition for how GraphRAG structures knowledge differently from standard RAG.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Color nodes by community
color_map = {}
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD', '#98D8C8', '#F7DC6F']
for comm_id, members in communities.items():
    for member in members:
        color_map[member] = colors[comm_id % len(colors)]

node_colors = [color_map.get(node, '#cccccc') for node in G.nodes()]

# Draw the graph
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=2, seed=42)

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold')
nx.draw_networkx_edges(G, pos, alpha=0.5, arrows=True, edge_color='gray')

# Add edge labels
edge_labels = nx.get_edge_attributes(G, 'label')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=6, alpha=0.7)

# Legend for communities
legend_patches = []
for comm_id, members in sorted(communities.items()):
    color = colors[comm_id % len(colors)]
    legend_patches.append(mpatches.Patch(color=color, label=f'Community {comm_id} ({len(members)} members)'))
plt.legend(handles=legend_patches, loc='upper left', fontsize=8)

plt.title("Knowledge Graph with Community Detection\n(Colors = Communities detected by Leiden algorithm)", fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

---

# Summary: Comparing the Three RAG Approaches

| Feature | Standard RAG | Agentic RAG | GraphRAG |
|---------|-------------|-------------|----------|
| **Retrieval method** | Vector similarity search | Agent decides when/what to retrieve | Graph traversal + community summaries |
| **Structure** | Flat chunks in a vector store | Tools + LLM reasoning loop | Knowledge graph with entities & edges |
| **Best for** | Factual, localized questions | Multi-step reasoning, tool use | Complex questions requiring connections across topics |
| **Handles "big picture" questions** | ❌ Poorly — no single chunk has the full answer | ⚠️ Can retry/reformulate, but still limited by chunk quality | ✅ Community summaries capture themes |
| **Can use external tools** | ❌ No | ✅ Yes — calculator, web search, APIs, etc. | ❌ Not by default (but can be combined with agents) |
| **Self-correcting** | ❌ No | ✅ Agent can reformulate if retrieval fails | ❌ Depends on graph quality |
| **Setup complexity** | Low | Medium | High (entity extraction, graph construction, community detection) |
| **Cost** | Low (one retrieval + one LLM call) | Medium (multiple LLM calls per tool use) | High (LLM calls for extraction, summarization, and answering) |

### When to use which?
- **Standard RAG**: Your go-to default. Works great for most Q&A over documents.
- **Agentic RAG**: When users ask multi-step questions, need calculations, or you want adaptive retrieval.
- **GraphRAG**: When you have highly interconnected data (org charts, research papers, legal documents) and need to answer thematic or cross-cutting questions.

### Further Reading
- [LangChain LCEL docs](https://python.langchain.com/docs/concepts/lcel/)
- [LangGraph ReAct Agent](https://langchain-ai.github.io/langgraph/tutorials/introduction/)
- [Microsoft GraphRAG Paper](https://arxiv.org/pdf/2404.16130)
- [Microsoft GraphRAG GitHub](https://github.com/microsoft/graphrag)